
# 🧠 LLMOps (Colab, End‑to‑End): Unsloth QLoRA → Push to Hugging Face → Serve with vLLM → Track with W&B

**What you'll do (student‑friendly & minimal):**
1. **Login** with your **Hugging Face token** (pull/push) **and** **W&B token** (experiment tracking).
2. Create a tiny **instruction dataset**, format with the model’s **chat template**.
3. **Fine‑tune** a small chat model with **Unsloth + QLoRA** (fast on T4).
4. **Merge** LoRA → base weights and **push** to **your** Hugging Face repo.
5. **Serve** with **vLLM** (OpenAI‑compatible, `--trust-remote-code`), then **validate** on a couple of prompts.
6. Track metrics, config, system stats, artifacts, and predictions in **Weights & Biases**.

> Default model: `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (lightweight for classrooms).



---

## 0) Runtime & GPU check


In [ ]:

import os, sys, torch
print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

BASE_DIR = "/content"
DATA_DIR = f"{BASE_DIR}/data"
OUT_DIR  = f"{BASE_DIR}/outputs"
MERGED_DIR = f"{OUT_DIR}/merged"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR,  exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print("Base model:", MODEL_ID)


Python : 3.12.12
PyTorch: 2.8.0+cu126
CUDA   : 12.6
GPU    : Tesla T4
Base model: TinyLlama/TinyLlama-1.1B-Chat-v1.0



---

## 1) 🔐 Log in: Hugging Face **and** Weights & Biases

- Create an HF token at https://huggingface.co/settings/tokens (scopes: **Read** + **Write**).
- Create a W&B token at https://wandb.ai/authorize.
- Replace with your own tokens.
- We log in programmatically and also export tokens to env so **subprocesses** (e.g., vLLM) can auth.


In [ ]:

from getpass import getpass
from huggingface_hub import login, whoami
import os, wandb

HF_TOKEN    = getpass("Paste your Hugging Face token (hidden): ")
WANDB_TOKEN = getpass("Paste your Weights & Biases API key (hidden): ")

assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "Please paste a valid HF token starting with 'hf_'!"
assert WANDB_TOKEN, "Please paste a valid W&B API key!"

# Login + export env so Transformers/hub/vLLM can use it
login(HF_TOKEN)
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN  # for vLLM/CLI subprocesses
os.environ["HUGGINGFACE_HUB_TOKEN"]  = HF_TOKEN  # set both variants for safety
os.environ["HF_TOKEN"]               = HF_TOKEN

# W&B login
wandb.login(key=WANDB_TOKEN)
os.environ["WANDB_API_KEY"] = WANDB_TOKEN
os.environ.setdefault("WANDB_PROJECT", "llmops_colab_tutorial")

me = whoami()
print("✅ HF user:", me.get("name", me))
print("✅ W&B logged in. Project:", os.environ["WANDB_PROJECT"])


Paste your Hugging Face token (hidden): ··········
Paste your Weights & Biases API key (hidden): ··········


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rajakarthik (rajakarthik-zenteiq) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your

✅ HF user: rajakarthik
✅ W&B logged in. Project: llmops_colab_tutorial



---

## 2) Install libraries

- `unsloth`, `transformers`, `datasets`, `peft`, `trl`, `bitsandbytes`
- **`wandb`** for tracking, **`vllm`** for serving


In [ ]:

%%bash
pip -q install -U pip
pip -q install -U "unsloth" "transformers>=4.44.0" "datasets>=2.19.0" \
    "accelerate>=0.34.0" "bitsandbytes>=0.43.0" "peft>=0.11.0" "trl>=0.9.6" \
    "huggingface_hub>=0.24.0" "wandb>=0.17.0"
# vLLM (may take a few minutes the first time)
pip -q install -U "vllm>=0.6.2"
python - <<'PY'
import torch
print("✅ Installs complete. torch:", torch.__version__)
PY


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.7 MB/s eta 0:00:00
✅ Installs complete. torch: 2.8.0+cu126


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.



---

## 3) Tiny instruction dataset → chat template

We keep it small for speed. In real projects, plug your dataset here.


In [ ]:
import json
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer

toy = [
    {"instruction": "Explain Apache Kafka in 2 lines.", "input": "", "output": "Kafka is a distributed event streaming platform. It lets apps publish/subscribe to event logs at massive scale."},
    {"instruction": "What is vLLM?", "input": "", "output": "vLLM is a high-throughput inference engine with an OpenAI-compatible API server."},
    {"instruction": "Summarize: Fine-tuning vs RAG in one sentence.", "input": "", "output": "Fine-tuning teaches the model new patterns; RAG retrieves external info at query time."},
    {"instruction": "Give one tip for faster training on small GPUs.", "input": "", "output": "Use QLoRA (4-bit) and small effective batch sizes with gradient accumulation."},
    {"instruction": "Write a friendly greeting for a student workshop.", "input": "", "output": "Hey there! Welcome to the LLM workshop—let’s build something awesome together!"},
    {"instruction": "Explain batching in one line.", "input": "", "output": "Batching groups multiple requests to improve GPU utilization and throughput."},
]

DATA_FILE = f"{DATA_DIR}/alpaca_tiny.jsonl"
with open(DATA_FILE, "w") as f:
    for row in toy:
        f.write(json.dumps(row) + "\n")
print("Wrote:", DATA_FILE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

def to_chat(example):
    user = example["instruction"] if not example.get("input") else f'{example["instruction"]}\n\n{example["input"]}'
    messages = [
        {"role":"user","content":user},
        {"role":"assistant","content":example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

raw = load_dataset("json", data_files={"train": DATA_FILE})["train"].shuffle(seed=42)
split = raw.train_test_split(test_size=0.34, seed=42)
train_ds = split["train"].map(to_chat)
val_ds   = split["test"].map(to_chat)
dset = DatasetDict({"train": train_ds, "validation": val_ds})
dset

Wrote: /content/data/alpaca_tiny.jsonl


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 3
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 3
    })
})


> **Why chat templates?** Hugging Face chat models expect a specific prompt format; using `apply_chat_template()` ensures correctness.



---

## 4) Fine‑tune with **Unsloth QLoRA** + **W&B tracking**

- Loads base in **4‑bit** (QLoRA path), attaches a **LoRA** adapter.  
- We set `report_to=["wandb"]` so **Trainer logs to W&B** automatically.  
- We also log the trained **adapter** as a W&B **artifact**.


In [ ]:
import os, torch, wandb, time
from unsloth import FastLanguageModel
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

RUN_NAME = f"unsloth_qlora_tinyllama_{int(time.time())}"
os.environ["WANDB_NAME"] = RUN_NAME

# Load base in 4-bit
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_ID,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    bias="none", task_type="CAUSAL_LM",
)
model = FastLanguageModel.get_peft_model(model,
    r = lora.r, # Pass r directly
    target_modules = lora.target_modules,
    lora_alpha = lora.lora_alpha,
    lora_dropout = lora.lora_dropout,
    bias = lora.bias,
    use_gradient_checkpointing = "unsloth",
)

args = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_seq_length=1024,
    logging_steps=10,
    save_steps=1000,
    eval_strategy="no",
    packing=True,
    bf16=False, # Set bf16 to False
    fp16=True,  # Enable fp16 training
    report_to=["wandb"],
    run_name=RUN_NAME,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dset["train"],
    args=args,
    dataset_text_field="text",
)

trainer.train()

ADAPTER_DIR = f"{OUT_DIR}/adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("✅ Saved LoRA adapter to:", ADAPTER_DIR)

# Log adapter as a W&B artifact
run = wandb.run or wandb.init(project=os.getenv("WANDB_PROJECT", "llmops_colab_tutorial"),
                              name=RUN_NAME, resume="allow", reinit=True)
artifact = wandb.Artifact("adapter-qlora", type="model")
artifact.add_dir(ADAPTER_DIR)
run.log_artifact(artifact)
run.config.update({"model_id": MODEL_ID, "max_seq_length": 1024, "learning_rate": 2e-4}, allow_val_change=True)
print("✅ Logged adapter artifact to W&B.")

/tmp/ipython-input-2181880987.py:2: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


INFO 10-27 03:16:04 [__init__.py:216] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.9: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.11.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/762M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 4,505,600 of 1,104,553,984 (0.41% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss


wandb: Adding directory to artifact (/content/outputs/adapter)... Done. 0.1s


✅ Saved LoRA adapter to: /content/outputs/adapter
✅ Logged adapter artifact to W&B.



---

## 5) Merge LoRA → base weights (easier for vLLM) and push to **your** Hub

- `merge_and_unload()` produces a single standard **Transformers** model.  
- We push **merged weights + tokenizer** to a new **private** repo and persist its name to a file.


In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig
from huggingface_hub import HfApi, create_repo, whoami
from datetime import datetime, timezone

# ====== MERGE ======
# Load base model and tokenizer
base_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load tokenizer from the base model (FIX: This was missing)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load and merge LoRA adapter
peft = PeftModel.from_pretrained(base_fp16, ADAPTER_DIR)
merged = peft.merge_and_unload()

# Save merged model and tokenizer
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("✅ Merged model saved to:", MERGED_DIR)

# ====== PUSH TO HUB ======
api = HfApi()
user = whoami()["name"]

# Fix deprecated datetime.utcnow()
repo_id = f"{user}/tinyllama-sft-colab-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}"
create_repo(repo_id=repo_id, private=True, exist_ok=True)

# Push model and tokenizer
merged.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

# Persist repo id for later cells
with open("/content/last_repo_id.txt", "w") as f:
    f.write(repo_id)

print("✅ Pushed merged model to:", repo_id)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Merged model saved to: /content/outputs/merged


...

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pgpcb1vgo/tokenizer.model: 100%|##########|  500kB /  500kB            

✅ Pushed merged model to: rajakarthik/tinyllama-sft-colab-20251027-031918



---

## 6) Serve with **vLLM** (OpenAI‑compatible, `--trust-remote-code`) and test

We run the server **in background**, then poll `GET /v1/models` until ready.  
If it fails, we show the last log lines for troubleshooting.


In [ ]:
# Run this cell to restart
import os
os.kill(os.getpid(), 9)


In [ ]:
import os, time, requests, subprocess, signal, psutil
import gc
import torch

# ====== CLEAR GPU MEMORY ======
gc.collect()
torch.cuda.empty_cache()

# Verify memory is freed
if torch.cuda.is_available():
    mem_free = torch.cuda.mem_get_info()[0] / 1024**3
    mem_total = torch.cuda.mem_get_info()[1] / 1024**3
    print(f"🔍 GPU Memory: {mem_free:.2f}GB free / {mem_total:.2f}GB total")

# Read repo id from file
with open("/content/last_repo_id.txt") as f:
    HF_REPO_ID = f.read().strip()
print("Serving HF repo:", HF_REPO_ID)

# Kill any existing vLLM processes
for p in psutil.process_iter(attrs=["pid", "name", "cmdline"]):
    try:
        cmd = " ".join(p.info["cmdline"] or [])
        if "vllm.entrypoints.openai.api_server" in cmd:
            os.kill(p.info["pid"], signal.SIGKILL)
            print(f"Killed existing vLLM process: {p.info['pid']}")
    except Exception:
        pass

# Setup environment tokens
env = os.environ.copy()
env["HUGGING_FACE_HUB_TOKEN"] = os.environ.get("HUGGING_FACE_HUB_TOKEN", "")
env["HUGGINGFACE_HUB_TOKEN"] = os.environ.get("HUGGINGFACE_HUB_TOKEN", "")
env["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")

# Launch vLLM with reduced memory utilization
cmd = [
    "nohup", "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", HF_REPO_ID,
    "--served-model-name", "my-served-model",
    "--port", "8000",
    "--trust-remote-code",
    "--gpu-memory-utilization", "0.70",  # Reduced from 0.85
    "--max-model-len", "2048",
]

print("Launching:", " ".join(cmd))
log_file = "/content/vllm_server.log"
with open(log_file, "w") as lf:
    proc = subprocess.Popen(cmd, stdout=lf, stderr=lf, env=env)

# Poll for readiness with extended timeout
base = "http://127.0.0.1:8000"
ready = False
for i in range(90):  # Increased from 60 to 90 attempts
    time.sleep(3)    # Increased from 2 to 3 seconds
    try:
        r = requests.get(base + "/v1/models", timeout=3)
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass

    # Show progress every 10 attempts
    if (i + 1) % 10 == 0:
        print(f"⏳ Still waiting... ({i + 1} attempts)")

print("\nReady:", ready)
if not ready:
    print("❌ vLLM not ready. Tail of logs:\n")
    try:
        print("\n".join(open(log_file).read().splitlines()[-200:]))
    except Exception as e:
        print("Could not read log:", e)
else:
    print("✅ vLLM is up and running!")


🔍 GPU Memory: 14.64GB free / 14.74GB total
Serving HF repo: rajakarthik/tinyllama-sft-colab-20251027-031918
Launching: nohup python -m vllm.entrypoints.openai.api_server --model rajakarthik/tinyllama-sft-colab-20251027-031918 --served-model-name my-served-model --port 8000 --trust-remote-code --gpu-memory-utilization 0.70 --max-model-len 2048
⏳ Still waiting... (10 attempts)
⏳ Still waiting... (20 attempts)
⏳ Still waiting... (30 attempts)
⏳ Still waiting... (40 attempts)
⏳ Still waiting... (50 attempts)
⏳ Still waiting... (60 attempts)
⏳ Still waiting... (70 attempts)
⏳ Still waiting... (80 attempts)

Ready: True
✅ vLLM is up and running!



---

## 7) Validate a couple prompts and log predictions to W&B

We hit the OpenAI‑compatible `/v1/chat/completions` and log a small **predictions table**.


In [ ]:

import requests, wandb

tests = [
    "Explain Apache Kafka in 2 lines.",
    "Give one tip for faster training on small GPUs."
]

def chat(user_text):
    payload = {
        "model": "my-served-model",
        "messages": [{"role":"user","content": user_text}],
        "temperature": 0.2,
        "max_tokens": 128,
    }
    r = requests.post("http://127.0.0.1:8000/v1/chat/completions", json=payload, timeout=60)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

rows = []
for q in tests:
    try:
        a = chat(q).strip()
        print("\nQ:", q, "\nA:", a)
        rows.append([q, a])
    except Exception as e:
        print("Request failed:", e)
        rows.append([q, f"ERROR: {e}"])

# Log a small predictions table to W&B
run = wandb.run or wandb.init(project=os.getenv("WANDB_PROJECT", "llmops_colab_tutorial"),
                              name=os.getenv("WANDB_NAME","unsloth_qlora_tinyllama"), resume="allow", reinit=True)
table = wandb.Table(columns=["question","answer"], data=rows)
run.log({"predictions": table})
print("✅ Logged predictions table to W&B.")



Q: Explain Apache Kafka in 2 lines. 
A: Apache Kafka is a distributed streaming platform that allows for real-time data processing and distribution. It provides a scalable, fault-tolerant, and distributed architecture that allows for the processing of large volumes of data in real-time. Kafka is designed to handle a wide range of data types, including structured, unstructured, and streaming data. It also supports a variety of programming languages and frameworks, including Java, Scala, Python, and Go. By using Kafka, developers can build scalable, fault-tolerant, and distributed applications that can handle large volumes of data in real-time.

Q: Give one tip for faster training on small GPUs. 
A: 1. Use GPU-accelerated libraries: GPU-accelerated libraries like CUDA, cuDNN, and TensorFlow can significantly speed up your training process on small GPUs. These libraries are designed to work with the GPU hardware and optimize the computation for the specific hardware.

2. Use smaller batc

wandb: Currently logged in as: rajakarthik (rajakarthik-zenteiq) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


✅ Logged predictions table to W&B.



---

## 8) (Optional) Stop vLLM


In [ ]:

import os, signal, psutil
for p in psutil.process_iter(attrs=["pid","name","cmdline"]):
    try:
        cmd = " ".join(p.info["cmdline"] or [])
        if "vllm.entrypoints.openai.api_server" in cmd:
            print("Killing", p.info["pid"])
            os.kill(p.info["pid"], signal.SIGKILL)
    except Exception:
        pass
print("Done.")


Killing 4745
Done.
